# 08 – Soru Metni, Güven Eşiği ve Son Model

6. haftanın sonunda üç iş planlamıştık. Bu not defteri üçünü de yapıyor:

1. **Soru metnini modele vermek:** Şimdiye kadar model sadece referans cevabı ve öğrenci cevabını okuyordu; **soruyu görmüyordu**. Örneğin "whenevery you dont specifiy your own" cevabı ancak soruyu bilen biri için anlamlı. Soruyu da modele verince, özellikle eğitimde hiç görülmemiş sorularda başarının artıp artmadığına bakacağız.
2. **Güven eşiğini sabitlemek:** 07'de en belirsiz cevapları öğretmene göndermenin hatayı azalttığını gördük. Ama gerçek sistemde "en belirsiz %25" diye değil, **sabit bir eşik** kullanmalıyız ("belirsizlik şu değerin üstündeyse öğretmene gönder"). Eşiği bir veride seçip **başka bir veride** sınayacağız.
3. **Son modeli eğitip kaydetmek:** Arayüzün ve puanlama servisinin kullanacağı modeli temiz verinin tamamıyla eğitip ayarlarıyla birlikte Drive'a kaydedeceğiz.

### Nasıl çalıştırılır?
1. **Önce GitHub'daki son pull request ana dala birleştirilmiş olmalı** (kavram analizi kodu `src/puanlama/kavram.py` orada).
2. **06 numaralı not defteri daha önce çalıştırılmış olmalı** (karşılaştırma için 06'nın sonuçları kullanılıyor).
3. Dosyayı Drive'daki `CevapAnaliz` klasörüne yükle, Colab'da aç, **T4 GPU**'yu seç ve **Tümünü çalıştır** de. Toplam süre yaklaşık **30 dakika**.

## 1. Kurulum

In [ ]:
import os, sys, subprocess
from pathlib import Path

COLAB = "google.colab" in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/CevapAnaliz")
    PROJE = Path("/content/CevapAnaliz")
    if PROJE.exists():
        subprocess.run(["git", "-C", str(PROJE), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Semantik-Muhendisleri/CevapAnaliz.git", str(PROJE)], check=True)
    (DRIVE / "data").mkdir(parents=True, exist_ok=True)
    if not (PROJE / "data").exists():
        os.symlink(DRIVE / "data", PROJE / "data")
else:  # Kendi bilgisayarında çalıştırılırsa
    PROJE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = PROJE

os.chdir(PROJE)
sys.path.insert(0, str(PROJE))
SEKIL = DRIVE / "sekiller"; SEKIL.mkdir(parents=True, exist_ok=True)
SONUC = DRIVE / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)

import json, time
import numpy as np
import pandas as pd
import torch
HIZLI_TEST = os.environ.get("CEVAPANALIZ_HIZLI_TEST") == "1"   # yalnızca geliştirici testi için
CIHAZ = "cuda" if torch.cuda.is_available() else "cpu"
if CIHAZ == "cpu" and not HIZLI_TEST:
    raise RuntimeError("GPU kapalı! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seçip baştan çalıştır.")
for dosya, mesaj in [(PROJE / "src" / "puanlama" / "kavram.py", "GitHub'daki pull request'i ana dala birleştir."),
                     (SONUC / "06_tahminler.csv", "Önce 06 numaralı not defterini çalıştır.")]:
    if not dosya.exists():
        raise RuntimeError(f"{dosya.name} bulunamadı. {mesaj}")

from src.veri.temizle import temizle
df, _ = temizle(pd.read_csv(PROJE / "data" / "processed" / "ortak_veri.csv", keep_default_na=False))
print("GPU:", torch.cuda.get_device_name(0) if CIHAZ == "cuda" else "yok (hızlı test)")
print(f"Temiz veri: {len(df):,} satır".replace(",", "."))

## 2. Model ve eğitim (06 ile aynı, tek fark: girdiye soru ekleniyor)

06'da modelin girdisi şuydu:  **[referans cevap] + [öğrenci cevabı]**
Bu not defterinde girdi şu oluyor:  **[soru + referans cevap] + [öğrenci cevabı]**

Model, ayarlar, test düzeni (Mohler'de sabit kuralla 5 grup; SciEntsBank ve Beetle'da temizlenmiş test kümeleri) ve belirsizlik hesabı (10 turluk Monte Carlo dropout) 06 ile birebir aynı. Böylece aradaki tek fark soru metni olur ve sonuçlar doğrudan karşılaştırılabilir.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

MODEL_ADI = "cross-encoder/stsb-distilroberta-base"
AYAR = {"epok": 3, "batch": 16, "ogrenme_orani": 2e-5, "max_uzunluk": 160, "mc_tur": 10}
if HIZLI_TEST:
    MODEL_ADI, AYAR["epok"], AYAR["mc_tur"] = "cross-encoder/stsb-TinyBERT-L4", 1, 3
tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
GPU = CIHAZ == "cuda"
SORU_EKLE = True

def kodla(satirlar):
    birinci = df.loc[satirlar, "referans_cevap"]
    if SORU_EKLE:
        birinci = df.loc[satirlar, "soru"] + " " + birinci
    return tokenizer(list(birinci), list(df.loc[satirlar, "ogrenci_cevap"]), truncation=True,
                     max_length=AYAR["max_uzunluk"], padding=True, return_tensors="pt").to(CIHAZ)

@torch.no_grad()
def puanla(model, satirlar):
    sonuc = []
    for i in range(0, len(satirlar), 128):
        with torch.autocast(device_type=CIHAZ, enabled=GPU):
            cikti = model(**kodla(satirlar[i:i + 128])).logits.squeeze(-1)
        sonuc.append(torch.sigmoid(cikti.float()).cpu().numpy())
    return np.concatenate(sonuc)

def tahmin_ve_belirsizlik(model, satirlar):
    model.eval(); tahmin = puanla(model, satirlar)
    model.train(); torch.manual_seed(0)
    turlar = np.stack([puanla(model, satirlar) for _ in range(AYAR["mc_tur"])])
    model.eval()
    return tahmin, turlar.std(axis=0)

def egit(egitim_satirlari, tohum=0):
    torch.manual_seed(tohum)
    rastgele = np.random.default_rng(tohum)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ADI).to(CIHAZ)
    optimizer = torch.optim.AdamW(model.parameters(), lr=AYAR["ogrenme_orani"], weight_decay=0.01)
    adim = AYAR["epok"] * int(np.ceil(len(egitim_satirlari) / AYAR["batch"]))
    zamanlayici = get_linear_schedule_with_warmup(optimizer, int(0.1 * adim), adim)
    olcekleyici = torch.amp.GradScaler(enabled=GPU)
    kayip_fonksiyonu = torch.nn.BCEWithLogitsLoss()
    for epok in range(AYAR["epok"]):
        model.train()
        sira, toplam = rastgele.permutation(egitim_satirlari), 0.0
        for i in range(0, len(sira), AYAR["batch"]):
            yigin = sira[i:i + AYAR["batch"]]
            puan = torch.tensor(df.loc[yigin, "puan"].values, dtype=torch.float32, device=CIHAZ)
            with torch.autocast(device_type=CIHAZ, enabled=GPU):
                kayip = kayip_fonksiyonu(model(**kodla(yigin)).logits.squeeze(-1).float(), puan)
            optimizer.zero_grad()
            olcekleyici.scale(kayip).backward()
            olcekleyici.step(optimizer); olcekleyici.update(); zamanlayici.step()
            toplam += kayip.item() * len(yigin)
        print(f"    epok {epok + 1}/{AYAR['epok']}: ortalama kayıp {toplam / len(sira):.4f}")
    return model

def egitim_planlari():
    mohler = df[df["veri_seti"] == "mohler"]
    grup = {s: i % 5 for i, s in enumerate(sorted(mohler["soru_id"].unique()))}   # 06 ile aynı sabit kural
    kat = mohler["soru_id"].map(grup)
    for k in range(5):
        yield f"Mohler (kat {k + 1}/5)", mohler.index[kat != k].values, {"Mohler": mohler.index[kat == k].values}
    for veri_seti, ad in [("scientsbank", "SciEntsBank"), ("beetle", "Beetle")]:
        secili = df["veri_seti"] == veri_seti
        yield ad, df.index[secili & (df["bolum"] == "train")].values, {
            f"{ad} – görülmemiş cevaplar": df.index[secili & (df["bolum"] == "test_ua")].values,
            f"{ad} – görülmemiş sorular": df.index[secili & (df["bolum"] == "test_uq")].values}

PLANLAR = list(egitim_planlari())
if HIZLI_TEST:
    PLANLAR = [(a, e[:256], {t: s[:96] for t, s in ts.items()}) for a, e, ts in PLANLAR]
TEST_SIRA = ["Mohler", "SciEntsBank – görülmemiş cevaplar", "SciEntsBank – görülmemiş sorular",
             "Beetle – görülmemiş cevaplar", "Beetle – görülmemiş sorular"]
QWK_SEVIYE = {t: (5 if t == "Mohler" else 2) for t in TEST_SIRA}
print("Model:", MODEL_ADI)

## 3. Soru metniyle eğitim (7 model, yaklaşık 18 dakika)

In [ ]:
kayitlar, baslangic = [], time.time()
for ad, egitim_satirlari, testler in PLANLAR:
    print(f"{ad}: {len(egitim_satirlari):,} cevapla eğitiliyor".replace(",", "."))
    model = egit(egitim_satirlari)
    for test_adi, test_satirlari in testler.items():
        tahmin, belirsizlik = tahmin_ve_belirsizlik(model, test_satirlari)
        kayitlar.append(pd.DataFrame({"test": test_adi, "tahmin": tahmin, "belirsizlik": belirsizlik}, index=test_satirlari))
    del model
    if GPU:
        torch.cuda.empty_cache()
print(f"Toplam süre: {(time.time() - baslangic) / 60:.1f} dakika")
soru_tahmin = pd.concat(kayitlar).join(df[["veri_seti", "bolum", "soru_id", "orijinal_puan", "puan"]])
soru_tahmin.to_csv(SONUC / "08_tahminler_soru.csv", encoding="utf-8")

## 4. Soru metni işe yaradı mı?

06'daki sonuçlarla (soru yok) karşılaştırıyoruz. Aynı model, aynı veri, aynı test düzeni; tek fark soru metni.

In [ ]:
from src.puanlama.olcutler import olc

onceki_tahmin = pd.read_csv(SONUC / "06_tahminler.csv", index_col=0, keep_default_na=False)
satirlar = []
for ad, tahmin in [("Soru yok (06)", onceki_tahmin), ("Soru metni eklendi (08)", soru_tahmin)]:
    for test_adi in TEST_SIRA:
        t = tahmin[tahmin["test"] == test_adi]
        if len(t):
            satirlar.append({"test": test_adi, "yontem": ad, **olc(t["puan"], t["tahmin"], QWK_SEVIYE[test_adi])})
karsilastirma = pd.DataFrame(satirlar)
karsilastirma.round(4).to_csv(SONUC / "08_soru_metni.csv", index=False, encoding="utf-8")
for olcut in ["pearson", "qwk"]:
    print(f"\n{olcut.upper()}")
    display(karsilastirma.pivot(index="yontem", columns="test", values=olcut).reindex(columns=TEST_SIRA).round(3))

ortalama = karsilastirma.groupby("yontem")["pearson"].mean()
SORU_EKLE_SON = bool(ortalama.get("Soru metni eklendi (08)", 0) > ortalama.get("Soru yok (06)", 0))
secilen_tahmin = soru_tahmin if SORU_EKLE_SON else onceki_tahmin
print(f"\nOrtalama Pearson: soru yok {ortalama.get('Soru yok (06)', float('nan')):.3f}, "
      f"soru eklendi {ortalama.get('Soru metni eklendi (08)', float('nan')):.3f}".replace(".", ","))
print("Son modelde soru metni", "KULLANILACAK." if SORU_EKLE_SON else "KULLANILMAYACAK.")

In [ ]:
import matplotlib.pyplot as plt

RENK = {"Soru yok (06)": "#2a78d6", "Soru metni eklendi (08)": "#eb6834"}
YONTEM = [y for y in RENK if y in set(karsilastirma["yontem"])]
fig, eksenler = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, olcut, baslik in zip(eksenler, ["pearson", "qwk"], ["Pearson korelasyonu", "QWK"]):
    for i, y in enumerate(YONTEM):
        d = karsilastirma[karsilastirma["yontem"] == y].set_index("test").reindex(TEST_SIRA)[olcut]
        ax.bar(np.arange(len(TEST_SIRA)) + (i - 0.5) * 0.38, d, width=0.35, color=RENK[y], label=y)
    ax.axhline(0.70, color="#52514e", linewidth=1, linestyle="--")
    ax.set_xticks(np.arange(len(TEST_SIRA)), [t.replace(" – ", "\n") for t in TEST_SIRA], fontsize=8.5)
    ax.set_title(baslik, loc="left", fontsize=12); ax.set_ylim(0, 1)
    for kenar in ["top", "right"]:
        ax.spines[kenar].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0"); ax.set_axisbelow(True)
tutamak, etiket = eksenler[0].get_legend_handles_labels()
fig.legend(tutamak, etiket, frameon=False, loc="upper left", ncol=2, bbox_to_anchor=(0.01, 0.93), fontsize=10)
fig.suptitle("Soru metnini modele vermenin etkisi", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(SEKIL / "08_soru_metni.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Güven eşiğini sabitlemek

Hedef: Gerçek kullanımda cevapların yaklaşık **%25'ini** öğretmene gönderecek **sabit bir belirsizlik eşiği** bulmak.

Eşiği seçtiğimiz veriyle sınarsak sonuç olduğundan iyi görünür (yine "kopya" sorunu). Bu yüzden:
1. Test sorularını rastgele **iki yarıya** ayırıyoruz (aynı sorunun cevapları aynı yarıda kalır).
2. Eşiği **1. yarıda** seçiyoruz (cevapların %25'inin üstünde kaldığı belirsizlik değeri) ve **2. yarıda** sınıyoruz; sonra yarıların yerini değiştirip tekrarlıyoruz.
3. 2. yarıda gerçekten yaklaşık %25 cevap öğretmene gidiyorsa ve hata azalıyorsa eşik güvenilirdir.

Mohler ile SciEntsBank/Beetle'ı ayrı ayrı ve birlikte inceliyoruz, çünkü veri setlerinin belirsizlik düzeyi farklı olabilir.

In [ ]:
rastgele = np.random.default_rng(0)
t = secilen_tahmin.copy()
t["hata"] = (t["tahmin"] - t["puan"]).abs()
sorular = t["soru_id"].unique()
yari = dict(zip(sorular, rastgele.permutation(len(sorular)) % 2))
t["yari"] = t["soru_id"].map(yari)

GRUPLAR = {"Mohler": t["veri_seti"] == "mohler", "SciEntsBank ve Beetle": t["veri_seti"] != "mohler",
           "Hepsi birlikte": pd.Series(True, index=t.index)}
satirlar = []
for ad, secim in GRUPLAR.items():
    g = t[secim]
    for secim_yarisi in [0, 1]:
        a, b = g[g["yari"] == secim_yarisi], g[g["yari"] != secim_yarisi]
        esik = a["belirsizlik"].quantile(0.75)
        giden = b["belirsizlik"] >= esik
        satirlar.append({"grup": ad, "eşik": esik, "sınanan yarıda giden oran": giden.mean(),
                         "MAE (hepsi)": b["hata"].mean(), "MAE (sistemde kalan)": b.loc[~giden, "hata"].mean(),
                         "MAE (öğretmene giden)": b.loc[giden, "hata"].mean()})
esik_sinama = pd.DataFrame(satirlar).groupby("grup", sort=False).mean()
esik_sinama["hata azalması (%)"] = (1 - esik_sinama["MAE (sistemde kalan)"] / esik_sinama["MAE (hepsi)"]) * 100
esik_sinama.round(4).to_csv(SONUC / "08_guven_esigi.csv", encoding="utf-8")
display(esik_sinama.round(3))

GUVEN_ESIGI = float(t["belirsizlik"].quantile(0.75))     # son model için: bütün test cevaplarından
GRUP_ESIKLERI = {ad: round(float(t.loc[secim, "belirsizlik"].quantile(0.75)), 4)
                 for ad, secim in GRUPLAR.items() if ad != "Hepsi birlikte"}
print("Veri setine özel eşikler (her birinde %25):", GRUP_ESIKLERI)
print(f"Son model için güven eşiği: {GUVEN_ESIGI:.4f}".replace(".", ","))
for ad, secim in GRUPLAR.items():
    print(f"  Bu eşikle {ad}: cevapların %{(t.loc[secim, 'belirsizlik'] >= GUVEN_ESIGI).mean() * 100:.0f}'i öğretmene gider")

**Tabloyu nasıl okuyalım?** "Sınanan yarıda giden oran" %25'e yakınsa, eşik başka sorulara da taşınabiliyor demektir. "Hata azalması" ise öğretmene giden cevaplar çıkarıldığında sistemde kalan cevaplarda hatanın yüzde kaç azaldığını gösterir. Veri setleri arasında belirsizlik düzeyi farklıysa tek bir ortak eşik, bir veri setinde %25'ten fazla, diğerinde az cevap gönderir; bu durumda sistemde eşiğin derse veya soru türüne göre ayarlanması gerekir.

## 6. Eksik kavram eşiği

07'deki yöntemle, SciEntsBank ve Beetle eğitim kümelerinden alınan cevaplarda en iyi kavram benzerlik eşiğini seçip son modelin ayarlarına kaydediyoruz.

In [ ]:
from scipy.stats import spearmanr
from src.puanlama.kavram import KavramAnalizci

analizci = KavramAnalizci(cihaz=CIHAZ)
n = 100 if HIZLI_TEST else 1500
ornek = pd.concat([df[(df["veri_seti"] == v) & (df["bolum"] == "train")].sample(n, random_state=0)
                   for v in ["scientsbank", "beetle"]])
benzerlikler = []
for referans, grup in ornek.groupby("referans_cevap", sort=False):
    for satir_no, sonuc in zip(grup.index, analizci.analiz(referans, list(grup["ogrenci_cevap"]))):
        benzerlikler.append((satir_no, [k["benzerlik"] for k in sonuc]))
benzerlikler = dict(benzerlikler)
puanlar = ornek["puan"]
ESIKLER = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
esik_tablosu = pd.Series({e: spearmanr([np.mean([b >= e for b in benzerlikler[i]]) for i in ornek.index], puanlar)[0]
                          for e in ESIKLER}, name="Spearman")
KAVRAM_ESIGI = float(esik_tablosu.idxmax())
display(esik_tablosu.round(3).to_frame())
print(f"Seçilen kavram eşiği: {KAVRAM_ESIGI:.2f}".replace(".", ","))

## 7. Son modelin eğitilmesi ve kaydedilmesi

Arayüzün ve puanlama servisinin kullanacağı model, temiz verideki **bütün eğitim cevaplarıyla** (Mohler'in tamamı, SciEntsBank ve Beetle eğitim kümeleri) eğitiliyor. Test kümeleri ileride yapılacak ölçümler için yine dışarıda bırakılıyor. Model, ayar dosyasıyla birlikte Drive'da `CevapAnaliz/modeller/ince_ayar_v1` klasörüne kaydediliyor.

In [ ]:
SORU_EKLE = SORU_EKLE_SON
egitim = df.index[(df["veri_seti"] == "mohler") | (df["bolum"] == "train")].values
if HIZLI_TEST:
    egitim = egitim[:256]
print(f"Son model {len(egitim):,} cevapla eğitiliyor (soru metni: {'var' if SORU_EKLE else 'yok'})".replace(",", "."))
son_model = egit(egitim)

KAYIT = DRIVE / "modeller" / "ince_ayar_v1"
KAYIT.mkdir(parents=True, exist_ok=True)
son_model.save_pretrained(KAYIT)
tokenizer.save_pretrained(KAYIT)
ayar = {"temel_model": MODEL_ADI, "soru_ekle": SORU_EKLE, "guven_esigi": round(GUVEN_ESIGI, 4),
        "guven_esigi_veri_setine_gore": GRUP_ESIKLERI, "kavram_esigi": KAVRAM_ESIGI, "egitim_cevap_sayisi": int(len(egitim)), "epok": AYAR["epok"],
        "tarih": time.strftime("%Y-%m-%d")}
(KAYIT / "ayar.json").write_text(json.dumps(ayar, ensure_ascii=False, indent=2), encoding="utf-8")
print("Kaydedildi:", KAYIT)
print(json.dumps(ayar, ensure_ascii=False, indent=2))

### Kaydedilen modelle kısa bir deneme

Puanlama modülünü (`src/puanlama/puanlayici.py`) kaydedilen modelle çalıştırıp birkaç cevabı puanlıyoruz: puan, belirsizlik, öğretmen incelemesi gerekip gerekmediği ve eksik kavramlar.

In [ ]:
from src.puanlama.puanlayici import CevapPuanlayici

del son_model
if GPU:
    torch.cuda.empty_cache()
puanlayici = CevapPuanlayici(KAYIT, cihaz=CIHAZ)
soru = "What stages in the software life cycle are influenced by the testing stage?"
referans = "The testing stage can influence both the coding stage (phase 5) and the solution refinement stage (phase 7)."
cevaplar = ["Testing influences coding and the refinement of the solution.",
            "The coding stage.",
            "It influences the design stage because errors are found.",
            "I don't know."]
for c, s in zip(cevaplar, puanlayici.puanla(referans, cevaplar, soru=soru)):
    print(f"{c}\n   puan {s['puan_5']:g}/5 · belirsizlik {s['belirsizlik']:.3f} · "
          f"{'ÖĞRETMEN İNCELEMESİ GEREKLİ · ' if s['inceleme_gerekli'] else ''}eksik: {s['eksik_kavramlar'] or '—'}")

## Özet

- **Soru metni** deneyi: Sonuçlar `sonuclar/08_soru_metni.csv` ve `sekiller/08_soru_metni.png`; son modelde kullanılıp kullanılmayacağına sonuçlara göre otomatik karar verildi.
- **Güven eşiği**: Eşik bir yarıda seçilip diğer yarıda sınandı (`sonuclar/08_guven_esigi.csv`); son model için eşik kaydedildi.
- **Kavram eşiği** eğitim verisinden seçildi.
- **Son model** ve ayarları Drive'da `CevapAnaliz/modeller/ince_ayar_v1` klasöründe.

### Arayüzü kendi bilgisayarında çalıştırmak için
1. Drive'daki `CevapAnaliz/modeller/ince_ayar_v1` klasörünü indir (sağ tık → İndir) ve zip'i aç.
2. Klasörü bilgisayarında proje içindeki `modeller/ince_ayar_v1` konumuna koy.
3. Proje klasöründe: `streamlit run src/arayuz/app.py`